# 4.2 · Hash functions

*Hashing* · 8 exercises

What makes a hash function spread keys well, and how picking it at random defeats hostile input.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-4.2) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-4.2) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-4.2)

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import random
import string
from collections import Counter

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    class _Raises:
        def __init__(self, kind):
            self.kind = kind

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            return issubclass(kind, self.kind)

    class pytest:  # noqa: N801
        raises = _Raises

### 4.2.1 · A naive string hash and its collisions  `Core`

**Task.** Write `sum_hash(s, m)` that adds the character codes and takes `% m`. Show that every anagram collides.

**Example.** `sum_hash("stop", 101) == sum_hash("pots", 101) == sum_hash("tops", 101)` -> `True`

**Big O.** O(len(s)). Ignoring position throws away most of the information in the key.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-4-2-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-4-2-1)

In [ ]:
# 4.2.1 A naive string hash and its collisions
def sum_hash(s, m):
    raise NotImplementedError

In [ ]:
# Test helpers: run once.
KEYS = range(0, 8000, 8)


def random_words():
    rng = random.Random(1)
    return {"".join(rng.choice(string.ascii_lowercase) for _ in range(6)) for _ in range(10000)}

In [ ]:
def test_ex_4_2_1():
    assert sum_hash("stop", 101) == sum_hash("pots", 101) == sum_hash("tops", 101)
    assert 0 <= sum_hash("anything", 7) < 7


test_ex_4_2_1()
print('4.2.1 passed')

### 4.2.2 · Polynomial string hashing  `Core`

**Task.** Write `poly_hash(s, m, base=31)` computing `(c0 * base^(L-1) + c1 * base^(L-2) + ... ) % m` with Horner's rule, so character order matters.

**Example.** `poly_hash("stop", 10**9 + 7) != poly_hash("pots", 10**9 + 7)` -> `True`

**Big O.** O(len(s)).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-4-2-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-4-2-2)

In [ ]:
# 4.2.2 Polynomial string hashing
def poly_hash(s, m, base=31):
    raise NotImplementedError

In [ ]:
def test_ex_4_2_2():
    assert poly_hash("stop", 10**9 + 7) != poly_hash("pots", 10**9 + 7)
    assert 0 <= poly_hash("anything", 7) < 7


test_ex_4_2_2()
print('4.2.2 passed')

### 4.2.3 · The division method and a bad table size  `Core`

**Task.** Write `division_hash(k, m)` = `k % m`. Hash the keys 0, 8, 16, ..., 7992 into m = 64 and into m = 61 and count how many buckets each uses.

**Example.** m = 64 -> only `8` buckets used; m = 61 (a prime) -> all `61` used

**Big O.** O(1) per key. Keys that share a factor with m crowd into few buckets; a prime m not near a power of two avoids the most common patterns.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-4-2-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-4-2-3)

In [ ]:
# 4.2.3 The division method and a bad table size
def division_hash(k, m):
    raise NotImplementedError


def buckets_used(keys, m, hash_fn):
    raise NotImplementedError

In [ ]:
def test_ex_4_2_3():
    assert buckets_used(KEYS, 64, division_hash) == 8 and buckets_used(KEYS, 61, division_hash) == 61
    assert division_hash(130, 64) == 2


test_ex_4_2_3()
print('4.2.3 passed')

### 4.2.4 · Measure a hash function's spread  `Core`

**Task.** Write `max_bucket(keys, m, hash_fn)` returning the size of the fullest bucket. A good function keeps it close to len(keys) / m.

**Example.** 10,000 words into 1,000 buckets with `poly_hash` -> a fullest bucket around 20 to 25; with `sum_hash`, many times more

**Big O.** O(N). The fullest bucket bounds the worst lookup time with chaining.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-4-2-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-4-2-4)

In [ ]:
# 4.2.4 Measure a hash function's spread
def max_bucket(keys, m, hash_fn):
    raise NotImplementedError

In [ ]:
def test_ex_4_2_4():
    words = random_words()
    assert max_bucket(words, 1000, poly_hash) < 30
    assert max_bucket(words, 1000, sum_hash) > 3 * max_bucket(words, 1000, poly_hash)


test_ex_4_2_4()
print('4.2.4 passed')

### 4.2.5 · Universal hashing  `Core`

**Task.** Write `UniversalHash(m, rng, p=2**61 - 1)` that picks random a in [1, p - 1] and b in [0, p - 1] once, and hashes an integer k as `((a * k + b) % p) % m`.

**Example.** two instances with different seeds send the same key to different buckets; any fixed pair of keys collides with probability about 1 / m

**Big O.** O(1) per key. Because the function is chosen at random after the keys are fixed, no input can be bad for most choices.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-4-2-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-4-2-5)

In [ ]:
# 4.2.5 Universal hashing
class UniversalHash:
    def __init__(self, m, rng, p=2**61 - 1):
        raise NotImplementedError

    def __call__(self, k):
        raise NotImplementedError

In [ ]:
def test_ex_4_2_5():
    h1, h2 = UniversalHash(1000, random.Random(1)), UniversalHash(1000, random.Random(2))
    assert any(h1(k) != h2(k) for k in range(10)) and all(0 <= h1(k) < 1000 for k in range(1000))


test_ex_4_2_5()
print('4.2.5 passed')

### 4.2.6 · Adversarial keys break a fixed function  `Practice`

**Task.** Write `adversarial_keys(m, count)` returning keys that all collide under `k % m` (multiples of m). Compare the fullest bucket under the division method and under `UniversalHash`.

**Example.** m = 1,000, 5,000 keys -> division method `5000` in one bucket; universal hash about `15`

**Big O.** Fixed function: O(N) per lookup on these keys. Universal: O(1 + N/m) expected on any keys.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-4-2-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-4-2-6)

In [ ]:
# 4.2.6 Adversarial keys break a fixed function
def adversarial_keys(m, count):
    raise NotImplementedError


def fullest(keys, fn):
    raise NotImplementedError

In [ ]:
def test_ex_4_2_6():
    adv = adversarial_keys(1000, 5000)
    assert len(adv) == 5000
    assert fullest(adv, lambda k: k % 1000) == 5000 and fullest(adv, UniversalHash(1000, random.Random(3))) < 25


test_ex_4_2_6()
print('4.2.6 passed')

### 4.2.7 · A hash table that re-randomizes when it grows  `Practice`

**Task.** Write `RandomizedHashTable(rng)` with chaining over integer keys: `put`, `get` and `__len__`. When the load factor passes 1, double the bucket count and draw a fresh `UniversalHash` before rehashing.

**Example.** insert 10,000 adversarial keys -> every value is found and the fullest bucket stays small

**Big O.** O(1) expected per operation, O(1) amortized for resizing.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-4-2-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-4-2-7)

In [ ]:
# 4.2.7 A hash table that re-randomizes when it grows
class RandomizedHashTable:
    def __init__(self, rng, capacity=8):
        raise NotImplementedError

    def _new_table(self, capacity):
        raise NotImplementedError

    def put(self, key, value):
        raise NotImplementedError

    def get(self, key, default=None):
        raise NotImplementedError

    def __len__(self):
        raise NotImplementedError

    def longest_chain(self):
        raise NotImplementedError

In [ ]:
def test_ex_4_2_7():
    table = RandomizedHashTable(random.Random(4))
    for k in adversarial_keys(1024, 10000):
        table.put(k, k + 1)
    assert len(table) == 10000 and all(table.get(k) == k + 1 for k in adversarial_keys(1024, 10000))
    assert table.longest_chain() < 12 and table.get(-5) is None
    table.put(0, "zero")
    assert table.get(0) == "zero" and len(table) == 10000


test_ex_4_2_7()
print('4.2.7 passed')

### 4.2.8 · The multiplication method  `Stretch`

**Task.** Write `mult_hash(k, m, A=0.6180339887)` = `floor(m * frac(k * A))` (Knuth's golden-ratio choice). Unlike the division method it works well even when m is a power of two.

**Example.** the keys 0, 8, ..., 7992 into m = 64 -> all `64` buckets used, where `k % 64` used only 8

**Big O.** O(1).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-4-2-8) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-4-2-8)

In [ ]:
# 4.2.8 The multiplication method
def mult_hash(k, m, A=0.6180339887498949):
    raise NotImplementedError

In [ ]:
def test_ex_4_2_8():
    assert buckets_used(KEYS, 64, mult_hash) == 64 and all(0 <= mult_hash(k, 64) < 64 for k in KEYS)


test_ex_4_2_8()
print('4.2.8 passed')